In [1]:
import time
import requests
from typing import Any, Dict, List, Optional


class PncpMergedClient:
    SEARCH_URL = "https://pncp.gov.br/api/search/"
    CONSULTA_URL = "https://pncp.gov.br/api/consulta/v1/contratacoes/proposta"

    def __init__(
        self,
        timeout: int = 60,
        sleep_seconds: float = 0.2,
        session: Optional[requests.Session] = None,
    ) -> None:
        self.timeout = timeout
        self.sleep_seconds = sleep_seconds
        self.session = session or requests.Session()
        self.session.headers.update(
            {
                "User-Agent": "Mozilla/5.0",
                "Accept": "application/json",
            }
        )

    # =========================
    # HTTP
    # =========================
    def _get(self, url: str, params: Dict[str, Any]) -> Any:
        response = self.session.get(url, params=params, timeout=self.timeout)
        print("[DEBUG URL]", response.url)

        if response.status_code == 400:
            return {"_http_400": True, "items": [], "data": []}

        response.raise_for_status()
        return response.json()

    # =========================
    # NORMALIZAÇÃO
    # =========================
    def _build_url(self, item_url: Optional[str]) -> Optional[str]:
        if not item_url:
            return None
        if item_url.startswith("http://") or item_url.startswith("https://"):
            return item_url
        return f"https://pncp.gov.br{item_url}"

    def _normalizar_search(self, item: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "numeroControlePNCP": item.get("numero_controle_pncp") or item.get("numeroControlePNCP"),
            "objetoCompra": item.get("description") or item.get("objetoCompra"),
            "titulo": item.get("title") or item.get("titulo"),
            "anoCompra": item.get("ano") or item.get("anoCompra"),
            "sequencialCompra": item.get("numero_sequencial") or item.get("sequencialCompra"),
            "numeroCompra": item.get("numero"),
            "processo": None,
            "modalidadeId": item.get("modalidade_licitacao_id") or item.get("modalidadeId"),
            "modalidadeNome": item.get("modalidade_licitacao_nome") or item.get("modalidadeNome"),
            "tipoInstrumentoConvocatorioCodigo": item.get("tipo_id") or item.get("tipoInstrumentoConvocatorioCodigo"),
            "tipoInstrumentoConvocatorioNome": item.get("tipo_nome") or item.get("tipoInstrumentoConvocatorioNome"),
            "situacaoCompraId": item.get("situacao_id") or item.get("situacaoCompraId"),
            "situacaoCompraNome": item.get("situacao_nome") or item.get("situacaoCompraNome"),
            "orgaoEntidade": {
                "cnpj": item.get("orgao_cnpj"),
                "razaoSocial": item.get("orgao_nome"),
                "poderId": item.get("poder_id"),
                "esferaId": item.get("esfera_id"),
            },
            "unidadeOrgao": {
                "codigoUnidade": item.get("unidade_codigo"),
                "nomeUnidade": item.get("unidade_nome"),
                "municipioNome": item.get("municipio_nome"),
                "ufSigla": item.get("uf"),
            },
            "dataPublicacaoPncp": item.get("data_publicacao_pncp") or item.get("dataPublicacaoPncp"),
            "dataAtualizacaoGlobal": item.get("data_atualizacao_pncp") or item.get("dataAtualizacaoPncp"),
            "dataAberturaProposta": item.get("data_inicio_vigencia"),
            "dataEncerramentoProposta": item.get("data_fim_vigencia"),
            "valorTotalEstimado": item.get("valor_global"),
            "valorTotalHomologado": None,
            "modoDisputaId": None,
            "modoDisputaNome": None,
            "linkSistemaOrigem": self._build_url(item.get("item_url") or item.get("itemUrl")),
            "linkProcessoEletronico": None,
            "fontesOrcamentarias": item.get("fonte_orcamentaria") or [],
            "usuarioNome": None,
            "_origem": "search",
            "_raw": item,
        }

    def _normalizar_consulta(self, item: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "numeroControlePNCP": item.get("numeroControlePNCP"),
            "objetoCompra": item.get("objetoCompra"),
            "titulo": item.get("tipoInstrumentoConvocatorioNome"),
            "anoCompra": item.get("anoCompra"),
            "sequencialCompra": item.get("sequencialCompra"),
            "numeroCompra": item.get("numeroCompra"),
            "processo": item.get("processo"),
            "modalidadeId": item.get("modalidadeId"),
            "modalidadeNome": item.get("modalidadeNome"),
            "tipoInstrumentoConvocatorioCodigo": item.get("tipoInstrumentoConvocatorioCodigo"),
            "tipoInstrumentoConvocatorioNome": item.get("tipoInstrumentoConvocatorioNome"),
            "situacaoCompraId": item.get("situacaoCompraId"),
            "situacaoCompraNome": item.get("situacaoCompraNome"),
            "orgaoEntidade": item.get("orgaoEntidade") or {},
            "unidadeOrgao": item.get("unidadeOrgao") or {},
            "dataPublicacaoPncp": item.get("dataPublicacaoPncp"),
            "dataAtualizacaoGlobal": item.get("dataAtualizacaoGlobal") or item.get("dataAtualizacao"),
            "dataAberturaProposta": item.get("dataAberturaProposta"),
            "dataEncerramentoProposta": item.get("dataEncerramentoProposta"),
            "valorTotalEstimado": item.get("valorTotalEstimado"),
            "valorTotalHomologado": item.get("valorTotalHomologado"),
            "modoDisputaId": item.get("modoDisputaId"),
            "modoDisputaNome": item.get("modoDisputaNome"),
            "linkSistemaOrigem": item.get("linkSistemaOrigem"),
            "linkProcessoEletronico": item.get("linkProcessoEletronico"),
            "fontesOrcamentarias": item.get("fontesOrcamentarias") or [],
            "usuarioNome": item.get("usuarioNome"),
            "_origem": "consulta",
            "_raw": item,
        }

    # =========================
    # REGRAS DE MERGE
    # =========================
    def _data_ref(self, item: Dict[str, Any]) -> str:
        return (
            item.get("dataAtualizacaoGlobal")
            or item.get("dataPublicacaoPncp")
            or item.get("dataEncerramentoProposta")
            or ""
        )

    def _is_empty(self, value: Any) -> bool:
        return value in (None, "", [], {})

    def _merge_dicts(self, base: Dict[str, Any], other: Dict[str, Any]) -> Dict[str, Any]:
        resultado = dict(base)

        for chave, valor in other.items():
            if chave not in resultado or self._is_empty(resultado[chave]):
                resultado[chave] = valor
                continue

            if isinstance(resultado[chave], dict) and isinstance(valor, dict):
                sub = dict(resultado[chave])
                for sk, sv in valor.items():
                    if sk not in sub or self._is_empty(sub[sk]):
                        sub[sk] = sv
                resultado[chave] = sub

        return resultado

    def _preferir_item(self, atual: Dict[str, Any], novo: Dict[str, Any]) -> Dict[str, Any]:
        escolhido = novo if self._data_ref(novo) > self._data_ref(atual) else atual
        outro = atual if escolhido is novo else novo

        merged = self._merge_dicts(escolhido, outro)

        origens = {atual.get("_origem"), novo.get("_origem")}
        if origens == {"search", "consulta"}:
            merged["_origem"] = "ambos"

        return merged

    # =========================
    # PAGINAÇÃO SEARCH
    # =========================
    def buscar_todas_paginas_search(
        self,
        tam_pagina: int = 100,
        tipos_documento: Optional[List[str]] = None,
        status: Optional[str] = None,
        ordenacao: str = "-data",
        max_paginas: Optional[int] = None,
        verbose: bool = True,
    ) -> List[Dict[str, Any]]:
        pagina = 1
        resultados: List[Dict[str, Any]] = []

        while True:
            if max_paginas is not None and pagina > max_paginas:
                if verbose:
                    print(f"[INFO] Limite de páginas do search atingido: {max_paginas}")
                break

            params: Dict[str, Any] = {
                "pagina": pagina,
                "tam_pagina": tam_pagina,
                "ordenacao": ordenacao,
            }

            if status:
                params["status"] = status

            if tipos_documento:
                params["tipos_documento"] = tipos_documento[0] if len(tipos_documento) == 1 else ",".join(tipos_documento)

            data = self._get(self.SEARCH_URL, params)

            if data.get("_http_400"):
                if verbose:
                    print(f"[INFO] Search retornou 400 na página {pagina}. Encerrando.")
                break

            items = data.get("items", [])

            if verbose:
                print(f"[INFO] Search página {pagina} | itens: {len(items)} | acumulado: {len(resultados)}")

            if not items:
                break

            resultados.extend([self._normalizar_search(x) for x in items])

            if len(items) < tam_pagina:
                break

            pagina += 1
            if self.sleep_seconds > 0:
                time.sleep(self.sleep_seconds)

        return resultados

    # =========================
    # PAGINAÇÃO CONSULTA
    # =========================
    def buscar_todas_paginas_consulta(
        self,
        data_final: int,
        data_inicial: Optional[int] = None,
        codigo_modalidade: Optional[int] = None,
        tamanho_pagina: int = 50,
        max_paginas: Optional[int] = None,
        verbose: bool = True,
    ) -> List[Dict[str, Any]]:
        pagina = 1
        resultados: List[Dict[str, Any]] = []

        while True:
            if max_paginas is not None and pagina > max_paginas:
                if verbose:
                    print(f"[INFO] Limite de páginas da consulta atingido: {max_paginas}")
                break

            params: Dict[str, Any] = {
                "dataFinal": data_final,
                "pagina": pagina,
                "tamanhoPagina": tamanho_pagina,
            }

            if data_inicial is not None:
                params["dataInicial"] = data_inicial

            if codigo_modalidade is not None:
                params["codigoModalidadeContratacao"] = codigo_modalidade

            data = self._get(self.CONSULTA_URL, params)

            if data.get("_http_400"):
                if verbose:
                    print(f"[INFO] Consulta retornou 400 na página {pagina}. Encerrando.")
                break

            items = data.get("data", [])

            if verbose:
                print(f"[INFO] Consulta página {pagina} | itens: {len(items)} | acumulado: {len(resultados)}")

            if not items:
                break

            resultados.extend([self._normalizar_consulta(x) for x in items])

            paginas_restantes = data.get("paginasRestantes")
            if paginas_restantes == 0:
                break

            if len(items) < tamanho_pagina:
                break

            pagina += 1
            if self.sleep_seconds > 0:
                time.sleep(self.sleep_seconds)

        return resultados

    # =========================
    # MERGE FINAL
    # =========================
    def buscar_e_mesclar(
        self,
        data_final_consulta: int,
        data_inicial_consulta: Optional[int] = None,
        codigo_modalidade_consulta: Optional[int] = None,
        tamanho_pagina_consulta: int = 50,
        tam_pagina_search: int = 100,
        tipos_documento_search: Optional[List[str]] = None,
        status_search: Optional[str] = "recebendo_proposta",
        ordenacao_search: str = "-data",
        max_paginas_search: Optional[int] = None,
        max_paginas_consulta: Optional[int] = None,
        verbose: bool = True,
    ) -> Dict[str, Any]:
        itens_search = self.buscar_todas_paginas_search(
            tam_pagina=tam_pagina_search,
            tipos_documento=tipos_documento_search,
            status=status_search,
            ordenacao=ordenacao_search,
            max_paginas=max_paginas_search,
            verbose=verbose,
        )

        itens_consulta = self.buscar_todas_paginas_consulta(
            data_final=data_final_consulta,
            data_inicial=data_inicial_consulta,
            codigo_modalidade=codigo_modalidade_consulta,
            tamanho_pagina=tamanho_pagina_consulta,
            max_paginas=max_paginas_consulta,
            verbose=verbose,
        )

        mapa: Dict[str, Dict[str, Any]] = {}
        sem_chave: List[Dict[str, Any]] = []

        for item in itens_search + itens_consulta:
            chave = item.get("numeroControlePNCP")

            if not chave:
                sem_chave.append(item)
                continue

            if chave in mapa:
                mapa[chave] = self._preferir_item(mapa[chave], item)
            else:
                mapa[chave] = item

        items_finais = list(mapa.values()) + sem_chave

        resumo_origem = {"search": 0, "consulta": 0, "ambos": 0}
        for item in mapa.values():
            origem = item.get("_origem")
            if origem in resumo_origem:
                resumo_origem[origem] += 1

        return {
            "resumo": {
                "total_search": len(itens_search),
                "total_consulta": len(itens_consulta),
                "total_unicos_com_chave": len(mapa),
                "total_sem_chave": len(sem_chave),
                "total_final": len(items_finais),
                "por_origem": resumo_origem,
            },
            "items": sorted(
                items_finais,
                key=lambda x: (
                    x.get("dataEncerramentoProposta") or "",
                    x.get("dataPublicacaoPncp") or "",
                    x.get("numeroControlePNCP") or "",
                )
            ),
        }

In [2]:
client = PncpMergedClient()

resultado = client.buscar_e_mesclar(
    data_final_consulta=20260319,
    data_inicial_consulta=20260301,      # opcional
    codigo_modalidade_consulta=None,     # opcional
    tamanho_pagina_consulta=50,
    tam_pagina_search=100,
    tipos_documento_search=["edital"],
    status_search="recebendo_proposta",
    ordenacao_search="-data",
    max_paginas_search=None,
    max_paginas_consulta=None,
    verbose=True,
)

print(resultado["resumo"])
print("Total final:", len(resultado["items"]))

[DEBUG URL] https://pncp.gov.br/api/search/?pagina=1&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital
[INFO] Search página 1 | itens: 100 | acumulado: 0
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=2&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital
[INFO] Search página 2 | itens: 100 | acumulado: 100
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=3&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital
[INFO] Search página 3 | itens: 100 | acumulado: 200
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=4&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital
[INFO] Search página 4 | itens: 100 | acumulado: 300
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=5&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital
[INFO] Search página 5 | itens: 100 | acumulado: 400
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=6&tam_pagina=100&orde

In [3]:
import json

with open("/Users/jose-cleiton/dev/script_pncp/pncp_merge_endpoints.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)

In [4]:
import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests


class PncpMergedProfessionalClient:
    """
    Cliente profissional para:
    - consumir diretamente os endpoints do PNCP
    - paginar search e consulta
    - fazer retry automático
    - normalizar registros
    - filtrar por palavras-chave
    - mesclar por numeroControlePNCP
    - salvar JSON e CSV
    """

    SEARCH_URL = "https://pncp.gov.br/api/search/"
    CONSULTA_URL = "https://pncp.gov.br/api/consulta/v1/contratacoes/proposta"
    BASE_SITE_URL = "https://pncp.gov.br"

    def __init__(
        self,
        timeout: int = 60,
        sleep_seconds: float = 0.2,
        max_retries: int = 4,
        backoff_factor: float = 1.5,
        session: Optional[requests.Session] = None,
    ) -> None:
        self.timeout = timeout
        self.sleep_seconds = sleep_seconds
        self.max_retries = max_retries
        self.backoff_factor = backoff_factor

        self.session = session or requests.Session()
        self.session.headers.update(
            {
                "User-Agent": "Mozilla/5.0",
                "Accept": "application/json",
            }
        )

    # =========================================================
    # HTTP / RETRY
    # =========================================================
    def _get(self, url: str, params: Dict[str, Any]) -> Any:
        tentativa = 0

        while True:
            try:
                response = self.session.get(
                    url,
                    params=params,
                    timeout=self.timeout,
                )

                print("[DEBUG URL]", response.url)

                if response.status_code == 400:
                    return {"_http_400": True, "items": [], "data": []}

                response.raise_for_status()
                return response.json()

            except requests.RequestException as e:
                tentativa += 1

                if tentativa > self.max_retries:
                    raise RuntimeError(
                        f"Falha após {self.max_retries} tentativas em {url} "
                        f"com params={params}. Erro: {e}"
                    ) from e

                espera = self.backoff_factor ** tentativa
                print(
                    f"[WARN] Erro na requisição (tentativa {tentativa}/{self.max_retries}). "
                    f"Aguardando {espera:.1f}s. Erro: {e}"
                )
                time.sleep(espera)

    # =========================================================
    # HELPERS
    # =========================================================
    def _build_url(self, item_url: Optional[str]) -> Optional[str]:
        if not item_url:
            return None
        if item_url.startswith("http://") or item_url.startswith("https://"):
            return item_url
        return f"{self.BASE_SITE_URL}{item_url}"

    def _is_empty(self, value: Any) -> bool:
        return value in (None, "", [], {})

    def _to_str(self, value: Any) -> str:
        if value is None:
            return ""
        return str(value)

    def _data_ref(self, item: Dict[str, Any]) -> str:
        return (
            item.get("dataAtualizacaoGlobal")
            or item.get("dataAtualizacao")
            or item.get("dataPublicacaoPncp")
            or item.get("dataEncerramentoProposta")
            or ""
        )

    def _normalizar_texto_busca(self, texto: str) -> str:
        return self._to_str(texto).strip().lower()

    def _match_keywords(
        self,
        texto: str,
        keywords_any: Optional[List[str]] = None,
        keywords_all: Optional[List[str]] = None,
    ) -> bool:
        texto_norm = self._normalizar_texto_busca(texto)

        if keywords_any:
            any_norm = [k.strip().lower() for k in keywords_any if k and k.strip()]
            if any_norm and not any(k in texto_norm for k in any_norm):
                return False

        if keywords_all:
            all_norm = [k.strip().lower() for k in keywords_all if k and k.strip()]
            if all_norm and not all(k in texto_norm for k in all_norm):
                return False

        return True

    # =========================================================
    # NORMALIZAÇÃO SEARCH
    # =========================================================
    def _normalizar_search(self, item: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "numeroControlePNCP": item.get("numero_controle_pncp") or item.get("numeroControlePNCP"),
            "objetoCompra": item.get("description") or item.get("objetoCompra"),
            "titulo": item.get("title") or item.get("titulo"),
            "anoCompra": item.get("ano") or item.get("anoCompra"),
            "sequencialCompra": item.get("numero_sequencial") or item.get("sequencialCompra"),
            "numeroCompra": item.get("numero"),
            "processo": None,
            "modalidadeId": item.get("modalidade_licitacao_id") or item.get("modalidadeId"),
            "modalidadeNome": item.get("modalidade_licitacao_nome") or item.get("modalidadeNome"),
            "tipoInstrumentoConvocatorioCodigo": item.get("tipo_id") or item.get("tipoInstrumentoConvocatorioCodigo"),
            "tipoInstrumentoConvocatorioNome": item.get("tipo_nome") or item.get("tipoInstrumentoConvocatorioNome"),
            "situacaoCompraId": item.get("situacao_id") or item.get("situacaoCompraId"),
            "situacaoCompraNome": item.get("situacao_nome") or item.get("situacaoCompraNome"),
            "orgaoEntidade": {
                "cnpj": item.get("orgao_cnpj"),
                "razaoSocial": item.get("orgao_nome"),
                "poderId": item.get("poder_id"),
                "esferaId": item.get("esfera_id"),
            },
            "unidadeOrgao": {
                "codigoUnidade": item.get("unidade_codigo"),
                "nomeUnidade": item.get("unidade_nome"),
                "municipioNome": item.get("municipio_nome"),
                "ufSigla": item.get("uf"),
            },
            "dataPublicacaoPncp": item.get("data_publicacao_pncp") or item.get("dataPublicacaoPncp"),
            "dataAtualizacaoGlobal": item.get("data_atualizacao_pncp") or item.get("dataAtualizacaoPncp"),
            "dataAberturaProposta": item.get("data_inicio_vigencia"),
            "dataEncerramentoProposta": item.get("data_fim_vigencia"),
            "valorTotalEstimado": item.get("valor_global"),
            "valorTotalHomologado": None,
            "modoDisputaId": None,
            "modoDisputaNome": None,
            "linkSistemaOrigem": self._build_url(item.get("item_url") or item.get("itemUrl")),
            "linkProcessoEletronico": None,
            "fontesOrcamentarias": item.get("fonte_orcamentaria") or [],
            "usuarioNome": None,
            "_origem": "search",
            "_raw": item,
        }

    # =========================================================
    # NORMALIZAÇÃO CONSULTA
    # =========================================================
    def _normalizar_consulta(self, item: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "numeroControlePNCP": item.get("numeroControlePNCP"),
            "objetoCompra": item.get("objetoCompra"),
            "titulo": item.get("tipoInstrumentoConvocatorioNome"),
            "anoCompra": item.get("anoCompra"),
            "sequencialCompra": item.get("sequencialCompra"),
            "numeroCompra": item.get("numeroCompra"),
            "processo": item.get("processo"),
            "modalidadeId": item.get("modalidadeId"),
            "modalidadeNome": item.get("modalidadeNome"),
            "tipoInstrumentoConvocatorioCodigo": item.get("tipoInstrumentoConvocatorioCodigo"),
            "tipoInstrumentoConvocatorioNome": item.get("tipoInstrumentoConvocatorioNome"),
            "situacaoCompraId": item.get("situacaoCompraId"),
            "situacaoCompraNome": item.get("situacaoCompraNome"),
            "orgaoEntidade": item.get("orgaoEntidade") or {},
            "unidadeOrgao": item.get("unidadeOrgao") or {},
            "dataPublicacaoPncp": item.get("dataPublicacaoPncp"),
            "dataAtualizacaoGlobal": item.get("dataAtualizacaoGlobal") or item.get("dataAtualizacao"),
            "dataAberturaProposta": item.get("dataAberturaProposta"),
            "dataEncerramentoProposta": item.get("dataEncerramentoProposta"),
            "valorTotalEstimado": item.get("valorTotalEstimado"),
            "valorTotalHomologado": item.get("valorTotalHomologado"),
            "modoDisputaId": item.get("modoDisputaId"),
            "modoDisputaNome": item.get("modoDisputaNome"),
            "linkSistemaOrigem": item.get("linkSistemaOrigem"),
            "linkProcessoEletronico": item.get("linkProcessoEletronico"),
            "fontesOrcamentarias": item.get("fontesOrcamentarias") or [],
            "usuarioNome": item.get("usuarioNome"),
            "_origem": "consulta",
            "_raw": item,
        }

    # =========================================================
    # MERGE
    # =========================================================
    def _merge_dicts(self, base: Dict[str, Any], other: Dict[str, Any]) -> Dict[str, Any]:
        resultado = dict(base)

        for chave, valor in other.items():
            if chave not in resultado or self._is_empty(resultado[chave]):
                resultado[chave] = valor
                continue

            if isinstance(resultado[chave], dict) and isinstance(valor, dict):
                sub = dict(resultado[chave])
                for subchave, subvalor in valor.items():
                    if subchave not in sub or self._is_empty(sub[subchave]):
                        sub[subchave] = subvalor
                resultado[chave] = sub

        return resultado

    def _preferir_item(self, atual: Dict[str, Any], novo: Dict[str, Any]) -> Dict[str, Any]:
        escolhido = novo if self._data_ref(novo) > self._data_ref(atual) else atual
        outro = atual if escolhido is novo else novo

        merged = self._merge_dicts(escolhido, outro)

        origens = {atual.get("_origem"), novo.get("_origem")}
        if origens == {"search", "consulta"}:
            merged["_origem"] = "ambos"

        return merged

    # =========================================================
    # SEARCH API
    # =========================================================
    def buscar_todas_paginas_search(
        self,
        tam_pagina: int = 100,
        tipos_documento: Optional[List[str]] = None,
        status: Optional[str] = None,
        ordenacao: str = "-data",
        max_paginas: Optional[int] = None,
        verbose: bool = True,
    ) -> List[Dict[str, Any]]:
        pagina = 1
        acumulado: List[Dict[str, Any]] = []

        while True:
            if max_paginas is not None and pagina > max_paginas:
                if verbose:
                    print(f"[INFO] Limite de páginas do search atingido: {max_paginas}")
                break

            params: Dict[str, Any] = {
                "pagina": pagina,
                "tam_pagina": tam_pagina,
                "ordenacao": ordenacao,
            }

            if status:
                params["status"] = status

            if tipos_documento:
                params["tipos_documento"] = (
                    tipos_documento[0]
                    if len(tipos_documento) == 1
                    else ",".join(tipos_documento)
                )

            data = self._get(self.SEARCH_URL, params)

            if data.get("_http_400"):
                if verbose:
                    print(f"[INFO] Search retornou 400 na página {pagina}. Encerrando.")
                break

            items = data.get("items", [])

            if verbose:
                print(f"[INFO] Search página {pagina} | itens: {len(items)} | acumulado: {len(acumulado)}")

            if not items:
                break

            acumulado.extend([self._normalizar_search(x) for x in items])

            if len(items) < tam_pagina:
                break

            pagina += 1
            if self.sleep_seconds > 0:
                time.sleep(self.sleep_seconds)

        return acumulado

    # =========================================================
    # CONSULTA API - INTERVALO DE DATAS
    # =========================================================
    def buscar_todas_paginas_consulta(
        self,
        data_inicial: Optional[int] = None,
        data_final: Optional[int] = None,
        codigo_modalidade: Optional[int] = None,
        tamanho_pagina: int = 50,
        max_paginas: Optional[int] = None,
        verbose: bool = True,
    ) -> List[Dict[str, Any]]:
        pagina = 1
        acumulado: List[Dict[str, Any]] = []

        while True:
            if max_paginas is not None and pagina > max_paginas:
                if verbose:
                    print(f"[INFO] Limite de páginas da consulta atingido: {max_paginas}")
                break

            params: Dict[str, Any] = {
                "pagina": pagina,
                "tamanhoPagina": tamanho_pagina,
            }

            if data_inicial is not None:
                params["dataInicial"] = data_inicial

            if data_final is not None:
                params["dataFinal"] = data_final

            if codigo_modalidade is not None:
                params["codigoModalidadeContratacao"] = codigo_modalidade

            data = self._get(self.CONSULTA_URL, params)

            if data.get("_http_400"):
                if verbose:
                    print(f"[INFO] Consulta retornou 400 na página {pagina}. Encerrando.")
                break

            items = data.get("data", [])

            if verbose:
                print(f"[INFO] Consulta página {pagina} | itens: {len(items)} | acumulado: {len(acumulado)}")

            if not items:
                break

            acumulado.extend([self._normalizar_consulta(x) for x in items])

            paginas_restantes = data.get("paginasRestantes")
            if paginas_restantes == 0:
                break

            if len(items) < tamanho_pagina:
                break

            pagina += 1
            if self.sleep_seconds > 0:
                time.sleep(self.sleep_seconds)

        return acumulado

    # =========================================================
    # FILTRO FINAL
    # =========================================================
    def filtrar_por_keywords(
        self,
        items: List[Dict[str, Any]],
        keywords_any: Optional[List[str]] = None,
        keywords_all: Optional[List[str]] = None,
    ) -> List[Dict[str, Any]]:
        if not keywords_any and not keywords_all:
            return items

        filtrados = []
        for item in items:
            texto = item.get("objetoCompra", "")
            if self._match_keywords(texto, keywords_any=keywords_any, keywords_all=keywords_all):
                filtrados.append(item)

        return filtrados

    # =========================================================
    # ORQUESTRAÇÃO
    # =========================================================
    def buscar_e_mesclar(
        self,
        # search
        tipos_documento_search: Optional[List[str]] = None,
        status_search: Optional[str] = "recebendo_proposta",
        ordenacao_search: str = "-data",
        tam_pagina_search: int = 100,
        max_paginas_search: Optional[int] = None,
        # consulta
        data_inicial_consulta: Optional[int] = None,
        data_final_consulta: Optional[int] = None,
        codigo_modalidade_consulta: Optional[int] = None,
        tamanho_pagina_consulta: int = 50,
        max_paginas_consulta: Optional[int] = None,
        # filtros finais
        keywords_any: Optional[List[str]] = None,
        keywords_all: Optional[List[str]] = None,
        # misc
        verbose: bool = True,
    ) -> Dict[str, Any]:
        itens_search = self.buscar_todas_paginas_search(
            tam_pagina=tam_pagina_search,
            tipos_documento=tipos_documento_search,
            status=status_search,
            ordenacao=ordenacao_search,
            max_paginas=max_paginas_search,
            verbose=verbose,
        )

        itens_consulta = self.buscar_todas_paginas_consulta(
            data_inicial=data_inicial_consulta,
            data_final=data_final_consulta,
            codigo_modalidade=codigo_modalidade_consulta,
            tamanho_pagina=tamanho_pagina_consulta,
            max_paginas=max_paginas_consulta,
            verbose=verbose,
        )

        unicos: Dict[str, Dict[str, Any]] = {}
        sem_chave: List[Dict[str, Any]] = []

        for item in itens_search + itens_consulta:
            chave = item.get("numeroControlePNCP")

            if not chave:
                sem_chave.append(item)
                continue

            if chave in unicos:
                unicos[chave] = self._preferir_item(unicos[chave], item)
            else:
                unicos[chave] = item

        items_finais = list(unicos.values()) + sem_chave

        items_finais = self.filtrar_por_keywords(
            items_finais,
            keywords_any=keywords_any,
            keywords_all=keywords_all,
        )

        items_finais = sorted(
            items_finais,
            key=lambda x: (
                x.get("dataEncerramentoProposta") or "",
                x.get("dataPublicacaoPncp") or "",
                x.get("numeroControlePNCP") or "",
            )
        )

        por_origem = {"search": 0, "consulta": 0, "ambos": 0}
        for item in items_finais:
            origem = item.get("_origem")
            if origem in por_origem:
                por_origem[origem] += 1

        return {
            "resumo": {
                "total_search": len(itens_search),
                "total_consulta": len(itens_consulta),
                "total_unicos_antes_filtro": len(unicos) + len(sem_chave),
                "total_final": len(items_finais),
                "total_sem_numero_controle": len(sem_chave),
                "keywords_any": keywords_any or [],
                "keywords_all": keywords_all or [],
                "por_origem": por_origem,
            },
            "items": items_finais,
        }

    # =========================================================
    # SAÍDA
    # =========================================================
    def salvar_json(self, resultado: Dict[str, Any], caminho_arquivo: str) -> None:
        caminho = Path(caminho_arquivo)
        caminho.parent.mkdir(parents=True, exist_ok=True)

        with open(caminho, "w", encoding="utf-8") as f:
            json.dump(resultado, f, ensure_ascii=False, indent=2)

        print(f"[OK] JSON salvo em: {caminho}")

    def salvar_csv(
        self,
        resultado: Dict[str, Any],
        caminho_arquivo: str,
        somente_colunas_principais: bool = False,
    ) -> None:
        caminho = Path(caminho_arquivo)
        caminho.parent.mkdir(parents=True, exist_ok=True)

        df = pd.json_normalize(resultado["items"])

        if somente_colunas_principais:
            colunas = [
                "numeroControlePNCP",
                "objetoCompra",
                "titulo",
                "anoCompra",
                "sequencialCompra",
                "numeroCompra",
                "processo",
                "modalidadeId",
                "modalidadeNome",
                "tipoInstrumentoConvocatorioCodigo",
                "tipoInstrumentoConvocatorioNome",
                "situacaoCompraId",
                "situacaoCompraNome",
                "orgaoEntidade.cnpj",
                "orgaoEntidade.razaoSocial",
                "orgaoEntidade.poderId",
                "orgaoEntidade.esferaId",
                "unidadeOrgao.codigoUnidade",
                "unidadeOrgao.nomeUnidade",
                "unidadeOrgao.municipioNome",
                "unidadeOrgao.ufSigla",
                "dataPublicacaoPncp",
                "dataAtualizacaoGlobal",
                "dataAberturaProposta",
                "dataEncerramentoProposta",
                "valorTotalEstimado",
                "valorTotalHomologado",
                "modoDisputaId",
                "modoDisputaNome",
                "linkSistemaOrigem",
                "linkProcessoEletronico",
                "usuarioNome",
                "_origem",
            ]
            colunas_existentes = [c for c in colunas if c in df.columns]
            df = df[colunas_existentes]

        df.to_csv(caminho, index=False, encoding="utf-8-sig")
        print(f"[OK] CSV salvo em: {caminho}")

In [ ]:
client = PncpMergedProfessionalClient(
    timeout=60,
    sleep_seconds=0.2,
    max_retries=4,
    backoff_factor=1.5,
)

resultado = client.buscar_e_mesclar(
    # search
    tipos_documento_search=["edital"],
    status_search="recebendo_proposta",
    ordenacao_search="-data",
    tam_pagina_search=100,
    max_paginas_search=None,

    # consulta
    data_inicial_consulta=20260301,
    data_final_consulta=20260319,
    codigo_modalidade_consulta=None,
    tamanho_pagina_consulta=50,
    max_paginas_consulta=None,

    # filtro final no objetoCompra
    keywords_any=["controle de ponto", "biometria", "controle de acesso", "reconhecimento facial"],
    keywords_all=None,

    verbose=True,
)

print(resultado["resumo"])

client.salvar_json(
    resultado,
    "/Users/jose-cleiton/dev/script_pncp/saida/pncp_merge_profissional.json"
)

client.salvar_csv(
    resultado,
    "/Users/jose-cleiton/dev/script_pncp/saida/pncp_merge_profissional.csv",
    somente_colunas_principais=True,
)